# Paper Figures: Weighted-Norm Bounds for MDP Approximation

This notebook reproduces all 10 paper figures (fig1–fig5) using the refactored modules.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
from models import InventoryMDP
from solvers import value_iteration, policy_evaluation
from bounds import weight_function, compute_kappa, compute_kappa_all, weighted_norm_bound_alpha_beta
from plots import (plot_single_bound_zoomed_out, plot_single_bound_zoomed_in,
                   plot_zoomed_out, plot_zoomed_in,
                   plot_alpha_beta_zoomed_out, plot_alpha_beta_zoomed_in)

# Build models
M = InventoryMDP(s_max=500, gamma=0.75, n=10, q=0.4, ch=4, cs=2, p=5)
M_hat = InventoryMDP(s_max=500, gamma=0.75, n=10, q=0.5, ch=3.8, cs=2, p=5)

# Value iteration
print("Running value iteration on M ...")
V_star, pi_star = value_iteration(M)
print("Running value iteration on M_hat ...")
V_hat_star, pi_hat_star = value_iteration(M_hat)

sigma_star = int(M.states[np.where(pi_star == 0)[0][0]])
sigma_hat = int(M_hat.states[np.where(pi_hat_star == 0)[0][0]])
print(f"Base-stock level sigma* = {sigma_star}")
print(f"Base-stock level sigma_hat* = {sigma_hat}")

# Policy evaluation
print("Running policy evaluation ...")
V_pi_hat_star = policy_evaluation(M, pi_hat_star)

gap = V_pi_hat_star - V_star
print(f"max(V_pi_hat_star - V_star) = {np.max(gap):.4f}")
print(f"min(V_pi_hat_star - V_star) = {np.min(gap):.6f}")

states = M.states

## Figures 1–2: Weighted vs Sup-Norm Bounds

In [ ]:
os.makedirs("../Figures/fig1", exist_ok=True)
os.makedirs("../Figures/fig2", exist_ok=True)

fig12_curves = []
fig12_labels = []

for ell, name in [(1.5e-2, "weight"), (0, "sup")]:
    weight = weight_function(M_hat, ell)
    kappa = max(
        compute_kappa(M, weight, pi_star),
        compute_kappa(M, weight, pi_hat_star),
        compute_kappa(M_hat, weight, pi_hat_star),
    )
    print(f"{name}: ell={ell:.2e}, kappa={kappa:.6f}")

    bound = weighted_norm_bound_alpha_beta(
        V_hat_star, pi_hat_star, weight, M, M_hat, kappa,
        alpha=1, beta=0,
    )
    bound_curve = bound * weight
    print(f"{name}: bound={bound:.4f}")

    label = r'$\ell = 0$' if ell == 0 else f'$\\ell = {ell:.1e}$'
    fig12_curves.append(bound_curve)
    fig12_labels.append(label)

    plot_single_bound_zoomed_out(
        states, V_pi_hat_star, bound_curve, label,
        filename="../Figures/fig1/" + name + "_zoomed_out.pdf")
    plot_single_bound_zoomed_in(
        states, V_pi_hat_star, bound_curve, label,
        filename="../Figures/fig2/" + name + "_zoomed_in.pdf")

print("Saved: fig1/ and fig2/")

In [ ]:
# Combined: both bounds on the same zoomed-out graph
plot_zoomed_out(states, V_pi_hat_star, fig12_curves, fig12_labels,
                filename="../Figures/fig1/combined_zoomed_out.pdf")
print("Saved: fig1/combined_zoomed_out.pdf")

## Figure 3: Multiple Weight Functions (Policy Stability)

In [ ]:
os.makedirs("../Figures/fig3", exist_ok=True)

l_arr = np.linspace(0, 2.5e-2, 6)
curves = []
labels = []

for ell in l_arr:
    weight = weight_function(M_hat, ell)
    kappa = max(
        compute_kappa(M, weight, pi_star),
        compute_kappa(M, weight, pi_hat_star),
        compute_kappa(M_hat, weight, pi_hat_star),
    )
    print(f"ell={ell:.4e}  kappa={kappa:.6f}")
    assert kappa < 1 / M.gamma, f"kappa={kappa:.4f} >= 1/gamma"

    bound = weighted_norm_bound_alpha_beta(
        V_hat_star, pi_hat_star, weight, M, M_hat, kappa,
        alpha=1, beta=0,
    )
    curves.append(bound * weight)

    if ell == 0:
        labels.append(r'$\ell = 0$')
    else:
        labels.append(f'$\\ell = {ell:.2e}$')

plot_zoomed_out(states, V_pi_hat_star, curves, labels,
                filename="../Figures/fig3/multi_out_500_new.pdf")
plot_zoomed_in(states, V_pi_hat_star, curves, labels,
               filename="../Figures/fig3/multi_in_500_new.pdf")
print("Saved: fig3/multi_out_500_new.pdf, fig3/multi_in_500_new.pdf")

## Figure 4: Alpha-Beta Bounds

In [ ]:
os.makedirs("../Figures/fig4", exist_ok=True)

ell = 1.5e-2
weight = weight_function(M_hat, ell)
kappa = max(
    compute_kappa(M, weight, pi_star),
    compute_kappa(M, weight, pi_hat_star),
    compute_kappa(M_hat, weight, pi_hat_star),
)
print(f"ell={ell:.2e}, kappa={kappa:.6f}")

configs = [
    (1, 0, r'$\alpha=1, \beta=0$'),
    (0.98, 0.8, r'$\alpha=0.98, \beta=0.8$'),
]

curves = []
labels = []
for alpha, beta, label in configs:
    bound = weighted_norm_bound_alpha_beta(
        V_hat_star, pi_hat_star, weight, M, M_hat, kappa,
        alpha=alpha, beta=beta,
    )
    curves.append(bound * weight)
    labels.append(label)
    print(f"alpha={alpha}, beta={beta}: bound={bound:.4f}")

plot_alpha_beta_zoomed_out(states, V_pi_hat_star, curves, labels,
                           filename="../Figures/fig4/alpha_beta_bound_out.pdf")
plot_alpha_beta_zoomed_in(states, V_pi_hat_star, curves, labels,
                          filename="../Figures/fig4/alpha_beta_bound_in.pdf")
print("Saved: fig4/alpha_beta_bound_out.pdf, fig4/alpha_beta_bound_in.pdf")

## Figure 5: Model Stability (All Actions)

In [ ]:
os.makedirs("../Figures/fig5", exist_ok=True)

l_arr = np.linspace(0, 2.5e-4, 6)
curves = []
labels = []

for ell in l_arr:
    weight = weight_function(M_hat, ell)
    kappa = max(
        compute_kappa_all(M, weight),
        compute_kappa_all(M_hat, weight),
    )
    if kappa >= 1 / M.gamma:
        print(f"ell={ell:.4e}  kappa={kappa:.6f}  SKIPPED (>= 1/gamma)")
        continue
    print(f"ell={ell:.4e}  kappa={kappa:.6f}")

    bound = weighted_norm_bound_alpha_beta(
        V_hat_star, pi_hat_star, weight, M, M_hat, kappa,
        alpha=1, beta=0,
    )
    curves.append(bound * weight)

    if ell == 0:
        labels.append(r'$\ell = 0$')
    else:
        labels.append(f'$\\ell = {ell:.2e}$')

plot_zoomed_out(states, V_pi_hat_star, curves, labels,
                filename="../Figures/fig5/multi_out_500_act_new.pdf")
plot_zoomed_in(states, V_pi_hat_star, curves, labels,
               filename="../Figures/fig5/multi_in_500_act_new.pdf")
print("Saved: fig5/multi_out_500_act_new.pdf, fig5/multi_in_500_act_new.pdf")